# Hackathon Task 3: Funkcje Feature Engineering
Notatnik ten przygotowuje kompleksowy potok przetwarzania danych (feature engineering). 

Zgodnie z wymaganiami przygotowaliśmy:
1. **Cechy wierszowe (Row-wise features)** - w tym cykliczne zmienne czasowe oraz różnice temperatur.
2. **Cechy szeregów czasowych (Time-series features)** - opóźnienia (lags) i średnie kroczące.
3. **Zmienne przestrzenne (Spatial features)** - łączenie urządzeń na podstawie `latitude` i `longitude` w klastry pogodowe/geograficzne.
4. **Integracja danych klimatycznych** - dedykowana struktura łączenia (mergowania) nowych danych.
5. **Agregację historii (Sequence/Snapshot)** - przekształcanie danych tak, by modelem przetwarzać całą historię urządzenia jako pojedynczy przypadek (wiersz) lub wszystkie urządzenia naraz z jednego dnia.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import warnings
import gc

warnings.filterwarnings('ignore')

## 1. Cechy Wierszowe i Czasowe (Row-wise & Time Variables)
Parsowanie czasu na informacje o danej porze dnia (co wpływa na obciążenie sieci `x2`) oraz tworzenie zmiennych różnicowych (Δ temperatur pomiędzy wymiennikami).

In [2]:
def create_time_features(df):
    """Tworzy zmienne czasowe z kolumny datetime."""
    df['hour'] = df['timedate'].dt.hour
    df['day_of_week'] = df['timedate'].dt.dayofweek
    df['day_of_month'] = df['timedate'].dt.day
    df['month'] = df['timedate'].dt.month
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Funkcje trygonometryczne dla reprezentacji cyklicznej czasu (zapobiega uskokom o północy)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    return df

def create_temp_diff_features(df):
    """Tworzy różnice temperatur (delty) w układzie. Wymienniki oddają ciepło do układu."""
    # Różnica temperatur na wymienniku dolnego źródła (Source HEX)
    df['t_diff_source'] = df['t3'] - df['t4']
    # Różnica temperatur na wymienniku obciążenia (Load HEX)
    df['t_diff_load1'] = df['t5'] - df['t6']
    df['t_diff_load2'] = df['t12'] - df['t13']
    # Różnica zewnętrzna vs wewnętrzna (Delta T)
    df['t_diff_ext_int'] = df['t1'] - df['t2']
    return df

## 2. Pamięć Szeregu Czasowego (Rolling Means & Lags)
Obliczanie przeszłości dla poszczególnej pompy (grupowane po `deviceId`). Zakładamy, że dane są posortowane rosnąco według `timedate`.

In [3]:
def create_time_series_features(df):
    """Tworzy opóźnienia i średnie kroczące opierając się na historycznych wierszach danej pompy."""
    # Sortowanie kluczowe do operacji okienkowych
    df = df.sort_values(by=['deviceId', 'timedate'])
    
    grouped = df.groupby('deviceId')
    
    # 1. Opóźnienia (Lags) z poprzedniego pomiaru (np. t-1 czyli -5 minut), t-12 (-1 godzina)
    df['x2_lag1'] = grouped['x2'].shift(1)  # Poprzednie 5 min
    df['x2_lag12'] = grouped['x2'].shift(12) # Godzinę temu
    df['t1_lag12'] = grouped['t1'].shift(12) # Temperatura zewn. godzinę temu
    
    # 2. Średnie kroczące (Rolling means) za ostatnie: 1h (12 kroków), 24h (288 kroków)
    df['x2_roll_mean_1h'] = grouped['x2'].rolling(window=12, min_periods=1).mean().reset_index(0, drop=True)
    df['t1_roll_mean_1h'] = grouped['t1'].rolling(window=12, min_periods=1).mean().reset_index(0, drop=True)
    
    # Odchylenie standardowe aby złapać lokalną zmienność pogody/zużycia
    df['x2_roll_std_1h'] = grouped['x2'].rolling(window=12, min_periods=1).std().reset_index(0, drop=True)
    # Wypełnienie ew. NaNs po lagu
    df.fillna(method='bfill', inplace=True)
    
    return df

## 3. Przetwarzanie Zmiennych Przestrzennych (Spatial & Devices Meta)
Łączenie urządzeń o podobnej lokalizacji ułatwia generalizację działania sieci, zacierając specyfikę jednej pompy.

In [4]:
def create_spatial_clusters(devices_df, n_clusters=10):
    """
    Klastruje urządzenia na podstawie ich szerokości i długości geograficznej.
    (Przygotowuje regiony pogodowe dla łatwiejszego złączenia w przyszłości)
    """
    coords = devices_df[['latitude', 'longitude']].dropna()
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    devices_df.loc[coords.index, 'geo_cluster'] = kmeans.fit_predict(coords)
    return devices_df

def merge_device_metadata(df, devices_df):
    """Spina metadane urządzenia (lub klaster) do głównej ramki szeregów czasowych"""
    return pd.merge(df, devices_df, on='deviceId', how='left')

## 4. Przygotowanie na wejście danych z pogody (Climate API/Data)
Funkcja, którą uruchomimy gdy tylko otrzymamy obiecane dane dotyczące klimatu.

In [5]:
def integrate_climate_data(df, climate_df):
    """
    Szkielet łączenia danych klimatycznych.
    Zależnie od ziarnistości (rozdzielczości) klimatu, łączymy po czasie oraz geografii (geo_cluster lub najbliższe coordynaty).
    Zakładamy, że climate_df posiada kolumny: ['timedate', 'geo_cluster', 'wind_speed', 'solar_irradiance', 'humidity'] itp.
    """
    # Należy upewnić się, że w obu ramkach czas jest zaokrąglony i odpowiada formatowi (np. do tej samej godziny lub dnia)
    climate_df['timedate'] = pd.to_datetime(climate_df['timedate'])
    
    # Złączenie danych głównej ramki df z climate_df 
    df_enriched = pd.merge(df, 
                           climate_df, 
                           on=['timedate', 'geo_cluster'], 
                           how='left')
    
    # Backfilling/Forward filling na wypadek różnicy fpekwencji (np pogoda co godzinę, grid co 5-min)
    df_enriched = df_enriched.fillna(method='ffill')
    
    return df_enriched

## 5. Zmiana formatu wejściowego pod Modele Niestandardowe
Zgodnie z wymaganiami użytkownika rozważamy dwa typy agregacji:
1. **Input całej historii konkretnej pompy w postaci wektora cech.**
2. **Input wszystkich pomp z danego dnia jako poszczególne cechy na jeden dzień (dla predykcji rynkowej / globalnej sieci).**

In [6]:
def create_daily_history_row(df):
    """
    Opcja 1: Tworzy jeden wiersz na urządzenie (lub urządzenie + miesiąc/rok),
    w którym historia z każdego dnia jest spłaszczona do nowych kolumn (np. day_1_mean, day_2_mean).
    """
    # Przykład operujący na średniej dziennej dla demonstracji (aby uniknąć miliarda kolumn przy 5-min interwale)
    df['date_only'] = df['timedate'].dt.date
    daily_agg = df.groupby(['deviceId', 'date_only'])[['x2', 't1']].mean().reset_index()
    
    # Pivot - czas (date_only) staje się kolumnami, a rzędem jest konkretne deviceId
    history_pivot = daily_agg.pivot(index='deviceId', columns='date_only', values=['x2', 't1'])
    
    # Spłaszczenie multi-indexu kolumn
    history_pivot.columns = [f"{col[0]}_{col[1]}" for col in history_pivot.columns]
    return history_pivot.reset_index()

def create_daily_snapshot_all_pumps(df):
    """
    Opcja 2: Traktujemy dany punkt czasu (np. dany dzień) jako wiersz (index),
    a stan poszczególnych pomp występuje jako wejściowe kolumny (feature'y). 
    Targetem jest wtedy zachowanie całej sieci dla poszczególnych dni.
    """
    df['date_only'] = df['timedate'].dt.date
    daily_agg = df.groupby(['date_only', 'deviceId'])['x2'].mean().reset_index()
    
    # Pivot - deviceId staje się kolumną
    snapshot_pivot = daily_agg.pivot(index='date_only', columns='deviceId', values='x2')
    snapshot_pivot.columns = [f"pump_{col}" for col in snapshot_pivot.columns]
    
    # Dodawanie globalnych cech (np. średnia t1 i x2 z całego kraju)
    global_agg = df.groupby('date_only')[['x2', 't1']].mean().reset_index()
    global_agg.rename(columns={'x2': 'global_avg_x2', 't1': 'global_avg_t1'}, inplace=True)
    
    snapshot_final = pd.merge(snapshot_pivot.reset_index(), global_agg, on='date_only')
    return snapshot_final

## Przykład Wywołania Całego Potoku
Gdy napłyną pełne dane i pogoda, wystarczy załadować ramkę danych (`df`) oraz uruchomić:
```python
devices = pd.read_csv('data/devices.csv')
devices = create_spatial_clusters(devices)

df = create_time_features(df)
df = create_temp_diff_features(df)
df = create_time_series_features(df)
df = merge_device_metadata(df, devices)
# Gdy napłyną dane pogodowe:
# df = integrate_climate_data(df, weather_df) 

# Jeśli chcemy wytrenować model LightGBM per pompa operujący na historii (agregacje):
# train_data = create_daily_history_row(df)

# Jeśli chcemy przewidzieć wszystkie pompy w danym dniu naraz:
# train_data = create_daily_snapshot_all_pumps(df)
```